In [2]:
!pip install gensim nltk pandas

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ------ --------------------------------- 3.9/24.4 MB 24.2 MB/s eta 0:00:01
   ------------------ --------------------- 11.0/24.4 MB 30.1 MB/s eta 0:00:01
   ---------------------------------- ----- 21.0/24.4 MB 36.8 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 36.7 MB/s  0:00:00

   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/2 [gensim]
   ---------------------------------------- 2/2 [gensim]



In [3]:
# For text preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Void2\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [4]:
documents = [
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
    "Rafael Nadal Is Out of the Australian Open",
    "Biden Announces Virus Measures",
    "Biden's Virus Plans Meet Reality",
    "Where Biden's Virus Plan Stands"
]

documents


['Rafael Nadal Joins Roger Federer in Missing U.S. Open',
 'Rafael Nadal Is Out of the Australian Open',
 'Biden Announces Virus Measures',
 "Biden's Virus Plans Meet Reality",
 "Where Biden's Virus Plan Stands"]

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    try:
        tokens = word_tokenize(text)
    except LookupError:
        # Backup tokenizer in case NLTK punkt is not available
        tokens = re.findall(r"\b[a-zA-Z0-9]+\b", text)

    tokens = [token for token in tokens if token.isalnum()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]
preprocessed_documents


[['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open'],
 ['rafael', 'nadal', 'australian', 'open'],
 ['biden', 'announces', 'virus', 'measure'],
 ['biden', 'virus', 'plan', 'meet', 'reality'],
 ['biden', 'virus', 'plan', 'stand']]

In [6]:
dictionary = corpora.Dictionary(preprocessed_documents)
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

print('Dictionary token count:', len(dictionary))
print('First document bag-of-words:', corpus[0])


Dictionary token count: 16
First document bag-of-words: [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1)]


In [7]:
lda_model = LdaModel(
    corpus,
    num_topics=2,
    id2word=dictionary,
    passes=15,
    random_state=42
)

lda_model.print_topics(num_words=6)


[(0,
  '0.131*"rafael" + 0.131*"nadal" + 0.131*"open" + 0.079*"federer" + 0.079*"roger" + 0.079*"join"'),
 (1,
  '0.167*"biden" + 0.166*"virus" + 0.119*"plan" + 0.071*"reality" + 0.071*"meet" + 0.071*"measure"')]

In [8]:
article_labels = []

for doc in preprocessed_documents:
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    article_labels.append(dominant_topic)

df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})
df_result


,Article,Topic
0,Rafael Nadal Joins Roger Federer in Missing U....,0
1,Rafael Nadal Is Out of the Australian Open,0
2,Biden Announces Virus Measures,1
3,Biden's Virus Plans Meet Reality,1
4,Where Biden's Virus Plan Stands,1


In [9]:
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()


Top terms for Topic #0:
['rafael', 'nadal', 'open', 'federer', 'roger', 'join', 'missing', 'australian', 'virus', 'biden']

Top terms for Topic #1:
['biden', 'virus', 'plan', 'reality', 'meet', 'measure', 'announces', 'stand', 'australian', 'open']



In [12]:
df = pd.read_csv('npr.csv')
documents = df['Article'].astype(str).tolist()

print('Number of articles:', len(documents))
df.head()


Number of articles: 11992


,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


In [13]:
preprocessed_documents = []

for i, doc in enumerate(documents):
    preprocessed_documents.append(preprocess_text(doc))
    if (i + 1) % 1000 == 0:
        print(f'Processed {i + 1} articles...')

print('Done')
print(preprocessed_documents[0][:50])


Processed 1000 articles...
Processed 2000 articles...
Processed 3000 articles...
Processed 4000 articles...
Processed 5000 articles...
Processed 6000 articles...
Processed 7000 articles...
Processed 8000 articles...
Processed 9000 articles...
Processed 10000 articles...
Processed 11000 articles...
Done
['washington', '2016', 'even', 'policy', 'bipartisan', 'politics', 'sense', 'year', 'show', 'little', 'sign', 'ending', 'president', 'obama', 'moved', 'sanction', 'russia', 'alleged', 'interference', 'election', 'concluded', 'republican', 'long', 'called', 'similar', 'severe', 'measure', 'could', 'scarcely', 'bring', 'approve', 'house', 'speaker', 'paul', 'ryan', 'called', 'obama', 'measure', 'appropriate', 'also', 'overdue', 'prime', 'example', 'administration', 'ineffective', 'foreign', 'policy', 'left', 'america', 'weaker']


In [14]:
dictionary = corpora.Dictionary(preprocessed_documents)
print('Before filtering:', len(dictionary))

dictionary.filter_extremes(no_below=15, no_above=0.5)
print('After filtering:', len(dictionary))

corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]
print('Number of documents in corpus:', len(corpus))


Before filtering: 86155
After filtering: 15974
Number of documents in corpus: 11992


In [ ]:
lda_model = LdaModel(
    corpus,
    num_topics=5,
    id2word=dictionary,
    passes=15,
    random_state=42,
    chunksize=2000
)

lda_model.print_topics(num_words=10)


In [ ]:
article_labels = []

for topics in lda_model[corpus]:
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    article_labels.append(dominant_topic)

df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})
df_result.to_csv('npr_topics_result.csv', index=False)

print('Saved results to npr_topics_result.csv')
df_result.head(20)


In [ ]:
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()


In [ ]:
print('Top Terms for Each Topic:')

for idx, topic in lda_model.print_topics(num_words=10):
    print(f'Topic {idx}:')
    terms = [term.strip() for term in topic.split('+')]
    for term in terms:
        weight, word = term.split('*')
        print(f'- {word.strip()} (weight: {weight.strip()})')
    print()


## Final discussion / interpretation

Use your actual top words to name the topics. The topic numbers may be different from another student because LDA is unsupervised.

Example possible topic labels:

| Topic clue words | Possible label |
|---|---|
| trump, clinton, election, republican, voters | US Election / Politics |
| health, care, medical, study, hospital | Health |
| school, students, education, university | Education |
| police, law, court, government | Law / Government |
| music, story, book, film, artist | Culture / Personal / Entertainment |

**Write-up:** The LDA model groups frequently co-occurring words into topics. After preprocessing the articles, a dictionary and corpus were created, then the LDA model identified five topics. The most meaningful words in each topic were inspected to assign human-readable labels such as politics, health, education, government/law, and culture/personal stories.